In [1]:
# --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. ---
# Z3ST: An open-source FEniCSx framework for thermo-mechanical analysis
# Author: Giovanni Zullo
# Version: 0.3.0 (2026)
# --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. ---

# Interactive 3D fields

Reads `output/fields.xdmf` through `plots.load_grid`. Stress and strain are in the local frame of the flats: nn through the wall, ss along the flat, zz along the axis.

Needs the trame backend: `pip install "pyvista[jupyter]"`.

In [2]:
import numpy as np
import pyvista as pv
import ipywidgets as widgets

from case_params import D, H
from plots import XDMF, load_grid
from z3st.utils.utils_extract_xdmf import list_steps_xdmf

pv.set_jupyter_backend("trame")
N_STEPS = len(list_steps_xdmf(XDMF, "Displacement"))


def show(field="T (K)", step=-1, deformed=True, warp=0.0, cut=True, edges=False):
    """One field on the original or deformed shape; ``warp=0`` draws the peak |u| as 5 % of D,
    ``cut`` keeps the lower half so the through-wall gradient shows on the cut face."""
    grid = load_grid(step)
    if deformed:
        u_max = max(np.linalg.norm(grid.point_data["u"], axis=1).max(), 1e-30)
        warp = warp or 0.05 * D / u_max
        mesh = grid.warp_by_vector("u", factor=warp)
    else:
        mesh = grid
    if cut:
        mesh = mesh.clip(normal="z", origin=(0, 0, H / 2))
    p = pv.Plotter()
    title = f"{field}, step {step}" + (f", deformed x{warp:.0f}" if deformed else "")
    p.add_text(title, font_size=10)
    if field is None:
        p.add_mesh(mesh, color="lightgray", show_edges=True, line_width=0.3)
    else:
        p.add_mesh(mesh, scalars=field, cmap="turbo", show_edges=edges,
                   scalar_bar_args={"title": field, "fmt": "%.4g"})
    if deformed:
        p.add_mesh(grid.clip(normal="z", origin=(0, 0, H / 2)) if cut else grid,
                   style="wireframe", color="black", opacity=0.05)
    p.view_isometric()
    p.show()

## Mesh

In [3]:
show(None, deformed=False, cut=False)

Widget(value='<iframe src="http://localhost:38357/index.html?ui=P_0x7f1da5e45700_0&reconnect=auto" class="pyvi…

## Original shape, temperature

In [4]:
show("T (K)", deformed=False)

Widget(value='<iframe src="http://localhost:38357/index.html?ui=P_0x7f1e506ec860_1&reconnect=auto" class="pyvi…

## Deformed shape, displacement magnitude

In [5]:
show("u (um)")

Widget(value='<iframe src="http://localhost:38357/index.html?ui=P_0x7f1d97faa7e0_2&reconnect=auto" class="pyvi…

## Stress

In [6]:
show("sigma_ss (MPa)")

Widget(value='<iframe src="http://localhost:38357/index.html?ui=P_0x7f1d97eae0c0_3&reconnect=auto" class="pyvi…

## Strain

In [7]:
show("eps_ss (-)")

Widget(value='<iframe src="http://localhost:38357/index.html?ui=P_0x7f1d97eacc80_4&reconnect=auto" class="pyvi…

## Explorer

Any field, step, warp and cut. Each change rebuilds the view.

In [8]:
g = load_grid()
fields = [k for k in g.point_data.keys() if k != "u"] + list(g.cell_data.keys())
widgets.interact(
    show,
    field=widgets.Dropdown(options=sorted(fields), value="sigma_ss (MPa)"),
    step=widgets.IntSlider(min=0, max=N_STEPS - 1, value=N_STEPS - 1),
    deformed=True,
    warp=widgets.FloatText(value=0.0, description="warp (0 = auto)"),
    cut=True,
    edges=False,
);

interactive(children=(Dropdown(description='field', index=5, options=('T (K)', 'eps_nn (-)', 'eps_ss (-)', 'ep…